<a href="https://colab.research.google.com/github/teederx/Trash-Classification-Using-Deeplearning/blob/main/Waste_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import kagglehub
import os

# Step 1: download and get local path
path = kagglehub.dataset_download("feyzazkefe/trashnet")

print(os.listdir(path))

# Step 3: load the csv using the path
# df = pd.read_csv(os.path.join(path, os.listdir(path)[0]))

100%|██████████| 40.8M/40.8M [00:00<00:00, 156MB/s]

Extracting files...


['dataset-resized']


In [24]:
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split, DataLoader

# Image transformations
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                     std=[0.5, 0.5, 0.5])
])

# Load dataset
dataset = ImageFolder(
    root=os.path.join(path, os.listdir(path)[0]),
    transform=transform
)

# 80% train, 20% test
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# print(f"Train samples: {len(train_dataset)}")
# print(f"Test samples: {len(test_dataset)}")
image, label = train_dataset[0]
image.size()

torch.Size([3, 64, 64])

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [30]:
class Net(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.feature_extractor = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Added another MaxPool2d layer

            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1), # Added another convolutional layer
            nn.BatchNorm2d(256),
            nn.ELU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 512),
            nn.ELU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [31]:
net = Net(num_classes=6)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
net.to(device)

Net(
  (feature_extractor): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ELU(alpha=1.0)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ELU(alpha=1.0)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True

In [33]:
for epoch in range(50):
    net.train()  # ensures layers like dropout/batchnorm behave correctly

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        # move data to GPU (faster training in Colab)
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()  # clear old gradients

        outputs = net(images)   # forward pass (prediction)
        loss = criterion(outputs, labels)  # compute error

        loss.backward()         # backpropagation (compute gradients)
        optimizer.step()        # update weight

        running_loss += loss.item()

        # accuracy calculation
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    scheduler.step()
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    print(f"Epoch {epoch+1} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.2f}%")

Epoch 1 | Loss: 1.2779 | Acc: 51.16%
Epoch 2 | Loss: 1.1784 | Acc: 55.32%
Epoch 3 | Loss: 1.1070 | Acc: 58.73%
Epoch 4 | Loss: 1.0633 | Acc: 60.91%
Epoch 5 | Loss: 1.0471 | Acc: 62.10%
Epoch 6 | Loss: 1.0189 | Acc: 63.09%
Epoch 7 | Loss: 0.9838 | Acc: 64.32%
Epoch 8 | Loss: 0.9416 | Acc: 64.87%
Epoch 9 | Loss: 0.9430 | Acc: 64.97%
Epoch 10 | Loss: 0.8606 | Acc: 67.94%
Epoch 11 | Loss: 0.8246 | Acc: 70.66%
Epoch 12 | Loss: 0.8060 | Acc: 70.21%
Epoch 13 | Loss: 0.7975 | Acc: 71.45%
Epoch 14 | Loss: 0.8117 | Acc: 71.40%
Epoch 15 | Loss: 0.8096 | Acc: 70.56%
Epoch 16 | Loss: 0.7816 | Acc: 71.60%
Epoch 17 | Loss: 0.7609 | Acc: 71.45%
Epoch 18 | Loss: 0.7572 | Acc: 72.24%
Epoch 19 | Loss: 0.7453 | Acc: 73.23%
Epoch 20 | Loss: 0.7143 | Acc: 74.62%
Epoch 21 | Loss: 0.6674 | Acc: 76.05%
Epoch 22 | Loss: 0.6701 | Acc: 76.89%
Epoch 23 | Loss: 0.6474 | Acc: 75.95%
Epoch 24 | Loss: 0.6641 | Acc: 74.72%
Epoch 25 | Loss: 0.6531 | Acc: 77.24%
Epoch 26 | Loss: 0.6448 | Acc: 77.04%
Epoch 27 | Loss: 0.64

In [36]:
torch.save(net.state_dict(), 'model.pth')

### Evaluate Model Performance on Test Set


In [37]:
net.eval()  # Set the model to evaluation mode

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():  # Disable gradient calculations during evaluation
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = net(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_test_loss = test_loss / len(test_loader)
final_test_acc = 100 * correct / total

print(f"Test Loss: {final_test_loss:.4f}")
print(f"Test Accuracy: {final_test_acc:.2f}%")

Test Loss: 0.7490
Test Accuracy: 73.72%


Load and test model on external data

In [41]:
from PIL import Image

In [54]:
new_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                     std=[0.5, 0.5, 0.5])
])

In [55]:
def load_image(image_path):
  image = Image.open(image_path)
  image = new_transform(image).float()
  image = image.unsqueeze(0)
  return image

In [78]:
image_path = '/content/new_images'
images = [load_image(os.path.join(image_path, img)) for img in os.listdir(image_path) if os.path.isfile(os.path.join(image_path, img))]

Evaluate model on new images

In [79]:
net.eval()
with torch.no_grad():
  for img in images:
    output = net(img)
    _,prediction = torch.max(output, 1)
    print(f'Prediction: {dataset.classes[prediction.item()]}')

Prediction: metal
Prediction: cardboard
